# Trabalho 1 — Caracterização e modelagem de uma rede complexa real

**Rede de aeroportos brasileiros**

Integrantes: preencher nomes do grupo

## 1. Introdução

Este trabalho investiga a estrutura da rede de voos domésticos regulares no Brasil em julho de 2026. Cada aeroporto é representado por um nó e cada rota por uma aresta direcionada. O peso da aresta é o número de voos observados na rota.

A pergunta central é: **quais propriedades estruturais tornam essa rede uma rede complexa e até que ponto elas podem ser reproduzidas por modelos clássicos?**

### 2. Dados e definições

Os dados são públicos e foram obtidos da ANAC:

- **VRA_20267.csv**: voos regulares ativos de julho de 2026;
- **AerodromosPublicos.csv**: cadastro de aeródromos públicos.

Um aeródromo é uma área destinada à operação de aeronaves. O código OACI identifica o aeródromo internacionalmente e o CIAD identifica o cadastro nacional. Neste estudo, os nós são os códigos OACI de origem e destino presentes nos voos filtrados.

### 3. Critérios de filtragem

Foram considerados apenas voos:

- domésticos: `Código Tipo Linha` igual a `N` ou `C`;
- regulares: `Código Autorização (DI)` igual a `0`, `4` ou `C`;
- não cancelados: `Situação Voo` diferente de `CANCELADO`;
- com origem e destino identificados por prefixos OACI brasileiros.

A rede é direcionada porque uma rota de A para B não implica necessariamente uma rota equivalente de B para A. Ela é ponderada porque cada aresta guarda o número de voos da rota.

### Acesso aos dados

Ambos disponíveis em:

- [ANAC Dados Abertos Voo Regular Ativo (VRA) Mensal](https://www.gov.br/anac/pt-br/acesso-a-informacao/dados-abertos/areas-de-atuacao/voos-e-operacoes-aereas/voo-regular-ativo-vra)
- [ANAC Dados Abertos Lista Aédromos](https://www.anac.gov.br/acesso-a-informacao/dados-abertos/areas-de-atuacao/aerodromos/lista-de-aerodromos-publicos-v2)

### 4. Leitura dos dados

Os arquivos ficam no mesmo diretório do notebook. A primeira linha dos arquivos contém metadados, por isso ela é ignorada na leitura.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

PASTA_DADOS = Path('.').resolve()
ARQUIVO_VRA = PASTA_DADOS / 'VRA_20267.csv'
ARQUIVO_AERODROMOS = PASTA_DADOS / 'AerodromosPublicos.csv'
prefixos_icao_brasil = ('SB', 'SD', 'SI', 'SJ', 'SN', 'SS', 'SW')

print(f'Pasta de trabalho: {PASTA_DADOS}')
print(f'VRA encontrado: {ARQUIVO_VRA.exists()}')
print(f'Aeródromos encontrado: {ARQUIVO_AERODROMOS.exists()}')

## 4.1 Dataframes originais

- `df_vra`: registros dos voos;
- `df_aerodromos`: cadastro e coordenadas dos aeródromos.

In [ ]:
df_vra = pd.read_csv(ARQUIVO_VRA, sep=';', skiprows=1, low_memory=False)
print(f'VRA: {df_vra.shape[0]:,} registros e {df_vra.shape[1]} colunas')
display(df_vra.head(2))

In [ ]:
df_aerodromos = pd.read_csv(
    ARQUIVO_AERODROMOS,
    sep=';',
    skiprows=1,
    low_memory=False,
    encoding='ISO-8859-1'
)
print(f'Aeródromos: {df_aerodromos.shape[0]:,} registros e {df_aerodromos.shape[1]} colunas')
display(df_aerodromos.head(2))

## 5. Filtragem e preparação

### 5.1 Voos

A filtragem segue os critérios definidos na seção anterior. Também removemos voos sem origem/destino e laços, pois o objetivo é estudar conexões entre aeroportos distintos.

In [ ]:
colunas_vra = [
    'Situação Voo',
    'Código Autorização (DI)',
    'Código Tipo Linha',
    'ICAO Aeródromo Origem',
    'ICAO Aeródromo Destino'
]

faltantes = sorted(set(colunas_vra) - set(df_vra.columns))
if faltantes:
    raise KeyError(f'Colunas ausentes no VRA: {faltantes}')

df_vra_filtrado = (
    df_vra
    .drop(columns=['Código Justificativa'], errors='ignore')
    .loc[
        lambda df: (
            (df['Situação Voo'] != 'CANCELADO')
            & df['Código Autorização (DI)'].isin(['0', '4', 'C'])
            & df['Código Tipo Linha'].isin(['N', 'C'])
            & df['ICAO Aeródromo Origem'].str.startswith(prefixos_icao_brasil, na=False)
            & df['ICAO Aeródromo Destino'].str.startswith(prefixos_icao_brasil, na=False)
        )
    ]
)

df_voos = df_vra_filtrado.dropna(
    subset=['ICAO Aeródromo Origem', 'ICAO Aeródromo Destino']
).copy()
df_voos = df_voos[
    df_voos['ICAO Aeródromo Origem'] != df_voos['ICAO Aeródromo Destino']
]

print(f'Antes: {len(df_vra):,} voos')
print(f'Após filtros: {len(df_voos):,} voos')
print(f'Removidos: {len(df_vra) - len(df_voos):,} ({(1 - len(df_voos) / len(df_vra)):.2%})')

### 5.2 Aeródromos

Para o cadastro, mantemos apenas aeródromos não interditados e com registro válido em julho de 2026. As colunas administrativas que não serão usadas são descartadas.

In [ ]:
df_aerodromos_filtrado = df_aerodromos.drop(
    columns=['Portaria de Registro', 'Link Portaria'],
    errors='ignore'
).copy()

df_aerodromos_filtrado = df_aerodromos_filtrado[
    df_aerodromos_filtrado['Situação'] != 'Interditado'
].copy()
df_aerodromos_filtrado['Validade do Registro'] = pd.to_datetime(
    df_aerodromos_filtrado['Validade do Registro'],
    format='%d/%m/%Y',
    errors='coerce'
)
data_comparacao = pd.Timestamp('2026-07-01')
df_aerodromos_filtrado = df_aerodromos_filtrado[
    df_aerodromos_filtrado['Validade do Registro'].isna()
    | (df_aerodromos_filtrado['Validade do Registro'] > data_comparacao)
].copy()

print(f'Antes: {len(df_aerodromos):,} aeródromos')
print(f'Após filtros: {len(df_aerodromos_filtrado):,} aeródromos')
print(f'Removidos: {len(df_aerodromos) - len(df_aerodromos_filtrado):,}')

## 6. Construção e caracterização estrutural da rede

As arestas são agregadas por origem e destino. O atributo `peso` registra o número de voos em cada rota.

### 6.1 Construção da representação computacional

In [ ]:
arestas_calculo = (
    df_voos
    .groupby(['ICAO Aeródromo Origem', 'ICAO Aeródromo Destino'])
    .size()
    .reset_index(name='peso')
)

G = nx.from_pandas_edgelist(
    arestas_calculo,
    source='ICAO Aeródromo Origem',
    target='ICAO Aeródromo Destino',
    edge_attr='peso',
    create_using=nx.DiGraph()
)
G_undirected = G.to_undirected()

print(G)
print(f'Rotas direcionadas distintas: {len(arestas_calculo):,}')
display(arestas_calculo.sort_values('peso', ascending=False).head(10))

### 6.2 Métricas estruturais

As métricas de caminhos, diâmetro e clustering são calculadas na projeção não direcionada. Para evitar resultados indefinidos, distância média e diâmetro usam a maior componente conexa.

In [ ]:
graus_totais = [grau for _, grau in G.degree()]
maior_componente = max(nx.connected_components(G_undirected), key=len)
G_core = G_undirected.subgraph(maior_componente).copy()

metricas_rede = {
    'Nós': G.number_of_nodes(),
    'Arestas direcionadas': G.number_of_edges(),
    'Arestas na projeção não direcionada': G_undirected.number_of_edges(),
    'Grau médio total': sum(graus_totais) / len(graus_totais),
    'Grau médio não direcionado': 2 * G_undirected.number_of_edges() / G_undirected.number_of_nodes(),
    'Densidade direcionada': nx.density(G),
    'Clustering médio': nx.average_clustering(G_undirected),
    'Componentes conexos': nx.number_connected_components(G_undirected),
    'Maior componente (nós)': G_core.number_of_nodes(),
    'Diâmetro da maior componente': nx.diameter(G_core),
    'Distância média da maior componente': nx.average_shortest_path_length(G_core),
}

for nome, valor in metricas_rede.items():
    formato = f'{valor:.4f}' if isinstance(valor, float) else f'{valor:,}'
    print(f'{nome}: {formato}')

## 7. Centralidade

As métricas abaixo representam conceitos diferentes de importância: conexões de entrada e saída, proximidade, intermediação, influência dos vizinhos e importância global.

In [ ]:
def top5(nome_metrica, centralidade):
    return (
        pd.Series(centralidade, name=nome_metrica)
        .sort_values(ascending=False)
        .head(5)
        .rename_axis('Aeroporto')
        .reset_index()
    )

centralidades = {
    'Entrada': nx.in_degree_centrality(G),
    'Saída': nx.out_degree_centrality(G),
    'Closeness': nx.closeness_centrality(G),
    'Betweenness': nx.betweenness_centrality(G),
    'PageRank': nx.pagerank(G),
    'Eigenvector': nx.eigenvector_centrality(G_undirected, max_iter=1000),
}

for nome, valores in centralidades.items():
    print(f'\nTop 5 — {nome}')
    display(top5(nome, valores))

In [ ]:
# Comparação visual entre centralidade de entrada e de saída
centralidade_grau = pd.DataFrame({
    'Entrada': centralidades['Entrada'],
    'Saída': centralidades['Saída'],
})
top_centralidade = centralidade_grau.mean(axis=1).nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
centralidade_grau.loc[top_centralidade.index].plot.barh(
    ax=ax,
    color=['#2563eb', '#f97316'],
    width=0.75
)
ax.set_xlabel('Centralidade de grau')
ax.set_ylabel('Aeroporto (OACI)')
ax.set_title('Aeroportos com maior centralidade média de grau')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()

## 8. Comparação com modelos clássicos

Os modelos têm o mesmo número de nós da projeção não direcionada real e aproximadamente o mesmo número de arestas. A semente fixa garante reprodutibilidade.

In [ ]:
seed = 42
nos = G_undirected.number_of_nodes()
arestas = G_undirected.number_of_edges()

G_random = nx.gnm_random_graph(n=nos, m=arestas, seed=seed)

k_medio = max(2, round(2 * arestas / nos))
k_medio = min(k_medio, nos - 1)
if k_medio % 2 != 0:
    k_medio -= 1
G_small_world = nx.watts_strogatz_graph(
    n=nos, k=k_medio, p=0.1, seed=seed
)

m_ba = max(1, min(round(arestas / nos), nos - 1))
G_scale_free = nx.barabasi_albert_graph(n=nos, m=m_ba, seed=seed)

def metricas_modelo(grafo):
    componente = max(nx.connected_components(grafo), key=len)
    grafo_core = grafo.subgraph(componente)
    return {
        'Densidade': nx.density(grafo),
        'Clustering': nx.average_clustering(grafo),
        'Diâmetro': nx.diameter(grafo_core),
        'Distância média': nx.average_shortest_path_length(grafo_core),
    }

modelos = {
    'Rede real': G_undirected,
    'Erdos-Renyi': G_random,
    'Watts-Strogatz': G_small_world,
    'Barabasi-Albert': G_scale_free,
}
resultados_modelos = pd.DataFrame({
    nome: metricas_modelo(grafo) for nome, grafo in modelos.items()
}).T

display(resultados_modelos.round(4))

### 8.1 Comparação das métricas e distribuição de graus

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(14, 9))
for eixo, metrica in zip(eixos.flat, resultados_modelos.columns):
    valores = resultados_modelos[metrica]
    barras = eixo.bar(valores.index, valores.values, color=['#2563eb', '#f97316', '#16a34a', '#dc2626'])
    eixo.bar_label(barras, fmt='%.3f' if metrica != 'Diâmetro' else '%.0f', padding=3, fontsize=8)
    eixo.set_title(metrica)
    eixo.tick_params(axis='x', labelrotation=25)
    eixo.grid(axis='y', alpha=0.25)
fig.suptitle('Comparação das métricas entre as redes', fontsize=14)
fig.tight_layout()
plt.show()

fig, eixos = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for eixo, (nome, grafo), cor in zip(
    eixos.flat,
    modelos.items(),
    ['#2563eb', '#f97316', '#16a34a', '#dc2626']
):
    distribuicao = pd.Series(dict(grafo.degree())).value_counts().sort_index()
    eixo.bar(distribuicao.index, distribuicao.values, color=cor, width=0.8)
    eixo.set_title(nome)
    eixo.set_xlabel('Grau')
    eixo.set_ylabel('Número de nós')
    eixo.grid(axis='y', alpha=0.25)
fig.suptitle('Distribuição de graus', fontsize=14)
fig.tight_layout()
plt.show()

## 9. Junção dos datasets e visualizações

O cadastro é unido duas vezes ao VRA: uma para enriquecer a origem e outra para enriquecer o destino.

In [ ]:
aero_cols = [
    'Código OACI', 'CIAD', 'Nome', 'Município', 'UF',
    'LATGEOPOINT', 'LONGEOPOINT'
]
df_aero = df_aerodromos_filtrado[aero_cols].drop_duplicates('Código OACI').copy()

df_aero_origem = df_aero.rename(columns={
    'Código OACI': 'Código OACI Origem',
    'CIAD': 'CIAD Origem',
    'Nome': 'Nome Origem',
    'Município': 'Município Origem',
    'UF': 'UF Origem',
    'LATGEOPOINT': 'Latitude Origem',
    'LONGEOPOINT': 'Longitude Origem',
})
df_aero_destino = df_aero.rename(columns={
    'Código OACI': 'Código OACI Destino',
    'CIAD': 'CIAD Destino',
    'Nome': 'Nome Destino',
    'Município': 'Município Destino',
    'UF': 'UF Destino',
    'LATGEOPOINT': 'Latitude Destino',
    'LONGEOPOINT': 'Longitude Destino',
})

df_vra_join = df_voos.merge(
    df_aero_origem,
    left_on='ICAO Aeródromo Origem',
    right_on='Código OACI Origem',
    how='left',
    validate='many_to_one'
).merge(
    df_aero_destino,
    left_on='ICAO Aeródromo Destino',
    right_on='Código OACI Destino',
    how='left',
    validate='many_to_one'
)

print(f'Voos antes das junções: {len(df_voos):,}')
print(f'Voos após as junções: {len(df_vra_join):,}')
display(df_vra_join.head(2))

In [ ]:
movimentacao = pd.concat([
    df_vra_join[['ICAO Aeródromo Origem', 'Nome Origem', 'Município Origem', 'UF Origem']].rename(columns={
        'ICAO Aeródromo Origem': 'OACI', 'Nome Origem': 'Nome',
        'Município Origem': 'Município', 'UF Origem': 'UF'
    }),
    df_vra_join[['ICAO Aeródromo Destino', 'Nome Destino', 'Município Destino', 'UF Destino']].rename(columns={
        'ICAO Aeródromo Destino': 'OACI', 'Nome Destino': 'Nome',
        'Município Destino': 'Município', 'UF Destino': 'UF'
    })
], ignore_index=True)

movimentacao['Aeroporto'] = (
    movimentacao['Nome']
    .fillna(movimentacao['Município'])
    .fillna(movimentacao['OACI'])
    .astype(str).str.strip().str.title()
    + ' (' + movimentacao['OACI'].astype(str).str.strip() + ')'
)
top_aeroportos = movimentacao['Aeroporto'].value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(11, 7))
top_aeroportos.plot.barh(ax=ax, color='#2563eb')
ax.bar_label(ax.containers[0], fmt='%d', padding=3)
ax.set_xlabel('Número de voos')
ax.set_ylabel('Aeroporto')
ax.set_title('Top 15 aeroportos por movimentação')
fig.tight_layout()
plt.show()

### 9.1 Rede geográfica

A visualização usa as coordenadas presentes no cadastro da ANAC. As arestas representam rotas, e os nós representam aeroportos.

In [ ]:
df_mapa = df_vra_join.dropna(
    subset=[
        'ICAO Aeródromo Origem', 'ICAO Aeródromo Destino',
        'Latitude Origem', 'Longitude Origem',
        'Latitude Destino', 'Longitude Destino'
    ]
).copy()

arestas_mapa = (
    df_mapa
    .groupby(['ICAO Aeródromo Origem', 'ICAO Aeródromo Destino'])
    .size()
    .reset_index(name='peso')
)
G_mapa = nx.from_pandas_edgelist(
    arestas_mapa,
    source='ICAO Aeródromo Origem',
    target='ICAO Aeródromo Destino',
    edge_attr='peso',
    create_using=nx.DiGraph()
)

posicoes = {}
for _, linha in df_mapa.iterrows():
    posicoes[linha['ICAO Aeródromo Origem']] = (
        linha['Longitude Origem'], linha['Latitude Origem']
    )
    posicoes[linha['ICAO Aeródromo Destino']] = (
        linha['Longitude Destino'], linha['Latitude Destino']
    )

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(
    G_mapa, posicoes, ax=ax, alpha=0.12, arrows=False, width=0.5
)
nx.draw_networkx_nodes(
    G_mapa, posicoes, ax=ax,
    node_size=[15 + 2 * G_mapa.degree(no) for no in G_mapa.nodes()],
    node_color='#dc2626', alpha=0.8
)
ax.set_xlim(-75, -34)
ax.set_ylim(-35, 6)
ax.set_title('Rede de voos domésticos — Julho/2026')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 10. Síntese para o relatório

Os resultados calculados nesta análise devem ser interpretados em conjunto:

- a rede filtrada representa voos domésticos regulares e não cancelados, e não todos os registros do VRA;
- os pesos preservam o volume de cada rota, enquanto as métricas topológicas calculadas acima consideram principalmente a existência das conexões;
- a comparação com os três modelos clássicos permite avaliar se a rede real combina baixa distância entre aeroportos, agrupamento local e concentração de conexões;
- a comparação entre as listas de centralidade ajuda a verificar se os aeroportos mais conectados também são os mais próximos, intermediários ou globalmente influentes;
- a maior componente deve ser usada ao interpretar diâmetro e distância média, pois a rede possui mais de um componente conexo.

### Perguntas para a conclusão

1. A rede real apresenta clustering maior ou menor que os modelos?
2. Qual modelo reproduz melhor a distância média e o diâmetro?
3. Os mesmos aeroportos aparecem nas diferentes métricas de centralidade?
4. A distribuição de graus sugere concentração de conexões em poucos aeroportos?
5. Quais limitações resultam do período de um único mês e do uso apenas de aeroportos públicos cadastrados?